# Chương 10. Hồi quy tuyến tính

Notebook đi kèm Chương 10 của nhánh AI Engineer trên trang *AI & Robotics*. Mỗi ô code ứng với một chỗ trong chương. Hãy đọc phần giải thích trên trang trước, rồi chạy ô tương ứng để kiểm chứng.

Cách chạy: bấm vào từng ô rồi nhấn **Shift + Enter**, hoặc chọn menu **Runtime > Run all**. Chương này không cần GPU.

## Trang 4 (Code): gradient descent tự viết bằng NumPy

Dữ liệu là Bảng 10.1: số giờ tự học mỗi tuần và điểm thi của 4 bạn sinh viên. Output phải khớp với output in trên trang, dòng cuối là `Bước 200 | w = 1.5993 | b = 0.5019 | MSE = 0.050001`.

In [ ]:
import numpy as np  # NumPy: thư viện tính toán trên cả mảng số cùng một lúc

# Dữ liệu Bảng 10.1. dtype=float vì lát nữa sẽ nhân chia ra số thập phân
x = np.array([1, 2, 3, 4], dtype=float)  # giờ tự học mỗi tuần của 4 bạn A, B, C, D
y = np.array([2, 4, 5, 7], dtype=float)  # điểm thi tương ứng của 4 bạn
n = len(x)                               # số mẫu dữ liệu, ở đây n = 4

w, b = 0.0, 0.0  # khởi tạo: chưa biết gì nên bắt đầu từ đường nằm ngang y = 0
lr = 0.05        # learning rate η, xem khung "Vì sao lại là η = 0,05?" ở trang Toán

for buoc in range(201):        # lặp 201 lần (bước 0 đến 200) để in được cả bước 0 lẫn bước 200
    y_hat = w * x + b          # đoán cho cả 4 bạn cùng lúc: NumPy nhân w với từng phần tử của x
    e = y - y_hat              # phần dư e_i = y_i - ŷ_i, cũng là một mảng 4 phần tử
    loss = np.mean(e ** 2)     # MSE, công thức (10.1): bình phương từng phần dư rồi lấy trung bình
    if buoc % 50 == 0:         # % là phép chia lấy dư: đúng ở bước 0, 50, 100, 150, 200
        print(f"Bước {buoc:3d} | w = {w:.4f} | b = {b:.4f} | MSE = {loss:.6f}")  # w, b in 4 chữ số thập phân, MSE in 6
    grad_w = -2 / n * np.sum(x * e)  # công thức (10.2): x * e nhân từng cặp phần tử rồi cộng lại
    grad_b = -2 / n * np.sum(e)      # công thức (10.3): cộng các phần dư
    w = w - lr * grad_w              # công thức (10.5): bước ngược hướng gradient
    b = b - lr * grad_b              # công thức (10.5) cho b, cùng độ dài bước lr

### Đối chiếu với scikit-learn

`scikit-learn` giải bằng nghiệm đóng, công thức (10.4). Kết quả phải trùng với gradient descent ở trên: w = 1,6 và b = 0,5.

In [ ]:
from sklearn.linear_model import LinearRegression  # mô hình hồi quy tuyến tính có sẵn của scikit-learn

X = x.reshape(-1, 1)          # sklearn cần X là bảng (số mẫu, số đặc trưng): đổi hình dạng từ (4,) thành (4, 1)
mo_hinh = LinearRegression()  # tạo mô hình rỗng, chưa học gì
mo_hinh.fit(X, y)             # "học": giải thẳng bằng nghiệm đóng, không lặp từng bước
print("w =", mo_hinh.coef_[0])      # coef_ là mảng các w, mỗi đặc trưng một số; ở đây chỉ có 1
print("b =", mo_hinh.intercept_)    # intercept_ là hệ số chặn b
print("R² =", mo_hinh.score(X, y))  # score trả về R², trang Đọc kết quả sẽ giải thích

## Trang 5 (Đọc kết quả): thử đổi learning rate

Ô này vẽ lại Hình 10.2. Hãy sửa các giá trị trong danh sách `cac_lr` rồi chạy lại để tự thấy ngưỡng phát nổ (trang Toán tính được là khoảng 0,12).

In [ ]:
import matplotlib.pyplot as plt  # thư viện vẽ biểu đồ

def chay_gd(lr, so_buoc=40):                # chạy gradient descent từ w = b = 0 với learning rate lr
    """Trả về danh sách MSE ở từng bước."""
    w = b = 0.0                             # cùng điểm xuất phát như ô ở trên
    lich_su = []                            # lưu MSE mỗi bước để vẽ
    for _ in range(so_buoc):                # dấu _ nghĩa là không cần dùng số thứ tự bước
        e = y - (w * x + b)                 # phần dư của đường thẳng hiện tại
        lich_su.append(np.mean(e ** 2))     # ghi lại MSE trước khi cập nhật
        w -= lr * (-2 / n * np.sum(x * e))  # w -= ... là viết gọn của w = w - ...
        b -= lr * (-2 / n * np.sum(e))      # cập nhật b tương tự
    return lich_su                          # trả về danh sách MSE

cac_lr = [0.005, 0.05, 0.135]  # quá nhỏ, vừa, quá lớn: thử sửa các số này
for lr_thu in cac_lr:                                              # chạy lần lượt từng learning rate
    plt.plot(chay_gd(lr_thu), label=f"learning rate = {lr_thu}")  # vẽ đường MSE theo bước lặp
plt.yscale("log")              # thang log vì MSE trải từ 0,05 đến hơn một tỷ
plt.xlabel("Bước lặp")         # tên trục ngang
plt.ylabel("MSE (thang log)")  # tên trục dọc
plt.legend()                   # hiện chú thích màu của từng đường
plt.show()                     # hiển thị hình

## Trang 6: gợi ý chạy bài tập

**Bài 10.2.** Trong ô gradient descent ở trên, đổi `if buoc % 50 == 0:` thành `if buoc <= 2:` để in ba bước đầu, rồi so với kết quả tính tay.

**Bài 10.3.** Trong cùng ô đó, đổi dòng dữ liệu thành `x = np.array([1, 2, 3, 4], dtype=float) * 60`. Đoán trước output, rồi chạy. Nhớ đổi lại như cũ trước khi chạy các ô khác.